Data ingestion pipeline - from Ingestion to VectorDB

In [2]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
#from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/home/zr/anaconda3/envs/rag311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def process_pdf(pdf_dir):
    all_docs = []
    pdf_dir = Path(pdf_dir)

    pdf_files = list(pdf_dir.glob("*.pdf"))
    print(f"found {len(pdf_files)} in pdf directory to process")

    for pdf_file in pdf_files:
        print(f"processing {pdf_file.name}")
        loader = PyMuPDFLoader(str(pdf_file))
        #loader = PyPDFLoader(str(pdf_file))
        docs = loader.load()
        for doc in docs:
            doc.metadata = {"source": pdf_file.name}
            doc.metadata = {"file_type": "pdf"}
        all_docs.extend(docs)
        print(f"loaded {len(docs)} pages from {pdf_file.name}")

    print(f"total docs loaded so far: {len(all_docs)}")
    return all_docs

all_pdf_docs = process_pdf("../data/pdf")

found 3 in pdf directory to process
processing dl-notes.pdf
loaded 58 pages from dl-notes.pdf
processing ml-notes.pdf
loaded 278 pages from ml-notes.pdf
processing agentic-ai-notes.pdf
loaded 28 pages from agentic-ai-notes.pdf
total docs loaded so far: 364


In [4]:
def split_documents(docs, chunk_size=800, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size, 
        chunk_overlap = chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(docs)
    print(f"split {len(docs)} documents into {len(split_docs)} chunks")
    
    return split_docs


In [5]:
chunks = split_documents(all_pdf_docs)
#chunks

split 364 documents into 1117 chunks


Embeddings and VectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer# embedding model
import uuid# id's for each chunk stored in VdB
import chromadb # vector database , faiss is another option for this
from chromadb.config import Settings
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        # This particular model is available on HuggingFace.
        # It converts chunks of text into embeddings(Vectors). 
        
        # Initialize the embedding manager
        
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
             print(f"Error loading model {self.model_name}: {e}")
             raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        #print("entered the generate_embeddings function")
        """
        Generate embeddings for a list of texts
        Arguments:
            texts: List of text strings to embed 
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        #if not self.model:
        #    raise ValueError("Model not loaded")
        #print("check the function run")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension: 384


In [ ]:
## Vector Database 
class VectorStore():
    def __init__(self, collection_name: str = "vector_store", persist_directory: str = "../data/vector_store"):
        # Initialize the vector store
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):